# Notebook 0 — Age Calculation for Longitudinal Data

This notebook computes precise chronological age at each visit for all
subjects in the OFAMS cohort. The resulting age table is used as input
in all subsequent notebooks.

Age is calculated by adding the fractional years elapsed since the baseline
visit to each subject's age at baseline, using exact visit dates from the
clinical database. The 10-year follow-up (m120) is handled separately as
visit dates for this timepoint are stored in a different source file.

## Contents

**0.1 Setup**
Imports and loading of raw visit date data from the clinical database.

**0.2 Preprocessing**
Alignment of subject ID columns across dataframes and conversion of visit
dates to datetime format.

**0.3 Age calculation**
Precise age at each visit computed as baseline age plus fractional years
elapsed since th Additionally, age at m120 is appended from a separate metadata file containing 10-year
follow-up information.
ed.csv` for use in Notebooks 1–4.


## 0.1 Setup

In [1]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load raw date data and save as CSV for portability
dates_df = pd.read_csv("dates.sav")
dates_df.to_csv("dates.csv", index=False)
dates_df.head()

C:\Users\Erlev\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\Erlev\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


,Unnamed: 0,Patnr,Visitnr,Visit_date,MRI_month,NT1Gd_dico,NT2_dico,CUA_dico,Serum_date,Month_serum,Vit_D,Vit_A,Vit_E,Days,Days_Pos
0,1,101,0,2004-12-28,12.0,1.0,NaN,NaN,2005-01-06,1.0,49.0,2.0,43.0,9.0,9.0
1,2,101,1,2005-01-27,1.0,1.0,1.0,1.0,2005-02-08,2.0,48.0,2.0,36.4,12.0,12.0
2,3,101,2,2005-02-28,2.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,101,3,2005-03-29,3.0,1.0,1.0,1.0,2005-05-11,5.0,53.0,1.9,29.7,43.0,43.0
4,5,101,4,2005-04-26,4.0,0.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
meta = pd.read_csv("follow_up_info.csv")
meta

,subject,sex,age_bl,age_10,EDSS_score_10
0,101,F,35.0,47.0,2.0
1,103,M,24.0,36.0,1.0
2,104,F,48.0,59.0,1.5
3,201,M,50.0,62.0,2.0
4,202,F,31.0,43.0,0.0
...,...,...,...,...,...
87,1501,F,46.0,58.0,3.0
88,1502,F,37.0,49.0,2.5
89,1503,M,45.0,57.0,4.0
90,1504,NaN,45.0,NaN,NaN


## 0.2 Preprocessing

In [9]:
# Ensure subject ID columns are integer type for consistent merging
dates_df['Patnr']    = dates_df['Patnr'].astype(int)
meta['subject']      = meta['subject'].astype(int)

# Convert visit date to datetime for age calculation
dates_df['Visit_date'] = pd.to_datetime(dates_df['Visit_date'])

In [11]:
def map_session_name(visitnr):
    """
    Map a numeric visit number to a session label.

    Parameters
    ----------
    visitnr : int or float
        Numeric visit identifier (0 = baseline, all others mapped to 'mN').

    Returns
    -------
    str
        Session label (e.g. 0 → 'baseline', 6 → 'm6').
    """
    if visitnr == 0:
        return 'baseline'
    else:
        return f'm{int(visitnr)}'

## 0.3 Calculate age

In [13]:
def calculate_precise_age(row, meta):
    """
    Calculate a subject's precise age at each visit.

    Age is computed by adding the time elapsed since the baseline visit
    (in fractional years) to the subject's age at baseline.

    Parameters
    ----------
    row : Series
        A single row from dates_df, containing 'Patnr' and 'Visit_date'.
    meta : DataFrame
        Metadata dataframe containing 'subject' and 'age_bl' (age at baseline).

    Returns
    -------
    float or None
        Precise age at the visit date, or None if subject or baseline
        date is not found.
    """
    sub_id       = row['Patnr']
    current_date = row['Visit_date']

    # Look up subject metadata
    sub_info = meta[meta['subject'] == sub_id]
    if sub_info.empty:
        return None

    age_bl = sub_info['age_bl'].values[0]

    # Find the subject's baseline visit date (Visitnr == 0)
    baseline_date = dates_df[
        (dates_df['Patnr'] == sub_id) &
        (dates_df['Visitnr'] == 0)
    ]['Visit_date']

    if baseline_date.empty:
        return None

    # Compute fractional years elapsed since baseline and add to baseline age
    delta_days = (current_date - baseline_date.iloc[0]).days
    return age_bl + (delta_days / 365.25)


# Compute precise age and map session labels for all visits
dates_df['age']     = dates_df.apply(lambda row: calculate_precise_age(row, meta), axis=1)
dates_df['session'] = dates_df['Visitnr'].apply(map_session_name)

In [17]:
# Build age table with standardized column names (bl to m24)
age_table = dates_df[['Patnr', 'session', 'age']].copy()
age_table.columns = ['subject', 'session', 'age']

In [19]:
# Add 10-year follow-up (m120) age from metadata
# m120 visit dates are stored separately in follow_up_info.csv rather than dates.sav
m120_rows = []
for _, row in meta.iterrows():
    if pd.notnull(row['age_10']):
        m120_rows.append({
            'subject': int(row['subject']),
            'session': 'm120',
            'age':     row['age_10']
        })

m120_df   = pd.DataFrame(m120_rows)
age_table = pd.concat([age_table, m120_df], ignore_index=True)

In [21]:
# Sort by subject and session order (baseline first, then chronologically)
age_table['sort_val'] = age_table['session'].apply(
    lambda x: 0 if x == 'baseline' else int(x[1:])
)
age_table = (age_table
             .sort_values(by=['subject', 'sort_val'])
             .drop(columns=['sort_val']))

# Save to CSV for use in subsequent notebooks
age_table.to_csv('Age_calculated.csv', index=False)
print("Saved: Age_calculated.csv")

# Spot check: display all sessions for one subject
print(age_table.query('subject == 101'))


Saved: Age_calculated.csv
      subject   session        age
0         101  baseline  35.000000
1         101        m1  35.082136
2         101        m2  35.169747
3         101        m3  35.249144
4         101        m4  35.325804
5         101        m5  35.407940
6         101        m6  35.498289
7         101        m7  35.594114
8         101        m8  35.670773
9         101        m9  35.750171
10        101       m12  35.999316
11        101       m18        NaN
12        101       m24  36.979466
1144      101      m120  47.000000
